# Type Hinting and Annotations

Python is **dynamically typed** — variables don't have fixed types. But you can add **type hints** as documentation:

```python
def greet(name: str) -> str:
    return f'Hello, {name}!'
```

Type hints:
- Document what types are expected
- Enable IDE autocompletion and error detection
- Can be checked statically with tools like `mypy`
- Are **NOT enforced at runtime** by default

By the end of this notebook you will be able to:
- Add type hints to functions and variables
- Use `Optional`, `Union`, `List`, `Dict`, `Tuple` etc.
- Use modern Python 3.10+ syntax
- Understand TypeVar and generics

## 1. Basic Type Hints

In [ ]:
# Variable annotations
name: str = 'Alice'
age: int = 30
score: float = 9.5
is_active: bool = True

print(name, age, score, is_active)

# Function annotations
def add(a: int, b: int) -> int:
    return a + b

def greet(name: str, title: str = 'Mr.') -> str:
    return f'Hello, {title} {name}!'

print(add(3, 4))
print(greet('Smith'))
print(greet('Johnson', 'Dr.'))

# Inspect annotations
print(add.__annotations__)

## 2. Collection Types

For built-in collections, use the type directly (Python 3.9+) or import from `typing`.

In [ ]:
# Python 3.9+ — use built-in types directly
def process_names(names: list[str]) -> list[str]:
    return [name.upper() for name in names]

def count_words(text: str) -> dict[str, int]:
    result: dict[str, int] = {}
    for word in text.split():
        result[word] = result.get(word, 0) + 1
    return result

def get_stats(numbers: list[float]) -> tuple[float, float, float]:
    """Returns (min, max, average)."""
    return min(numbers), max(numbers), sum(numbers)/len(numbers)

print(process_names(['alice', 'bob', 'charlie']))
print(count_words('the quick brown fox the fox'))
print(get_stats([1.0, 2.5, 3.7, 4.2]))

In [ ]:
# For Python < 3.9, import from typing
from typing import List, Dict, Tuple, Set

def old_style(names: List[str]) -> Dict[str, int]:
    return {name: len(name) for name in names}

print(old_style(['Alice', 'Bob', 'Charlie']))

## 3. `Optional` — Value That Might Be None

In [ ]:
from typing import Optional

# Optional[X] = Union[X, None] = the value can be X or None
def find_user(user_id: int) -> Optional[dict]:
    users = {1: {'name': 'Alice'}, 2: {'name': 'Bob'}}
    return users.get(user_id)  # returns dict or None

user = find_user(1)
if user:  # must check for None
    print(user['name'])

missing = find_user(99)
print(missing)  # None

# Python 3.10+ syntax: X | None
def find_user_modern(user_id: int) -> dict | None:
    users = {1: {'name': 'Alice'}}
    return users.get(user_id)

print(find_user_modern(1))
print(find_user_modern(99))

## 4. `Union` — Multiple Possible Types

In [ ]:
from typing import Union

# Old style
def stringify(value: Union[int, float, str]) -> str:
    return str(value)

# Python 3.10+ (preferred)
def stringify_modern(value: int | float | str) -> str:
    return str(value)

print(stringify(42))
print(stringify(3.14))
print(stringify('hello'))

# Flexible function
def parse_number(text: str) -> int | float:
    try:
        return int(text)
    except ValueError:
        return float(text)

print(parse_number('42'))   # int
print(parse_number('3.14')) # float

## 5. `Any` — Opt Out of Type Checking

In [ ]:
from typing import Any

# Use Any when the type is genuinely flexible
def log(message: str, data: Any = None) -> None:
    print(f'[LOG] {message}', end='')
    if data is not None:
        print(f': {data}', end='')
    print()

log('User created', {'name': 'Alice'})
log('Count', 42)
log('Error occurred')

## 6. `Callable` — Type Hints for Functions

In [ ]:
from typing import Callable

# Callable[[arg_types], return_type]
def apply(func: Callable[[int], int], value: int) -> int:
    return func(value)

def apply_many(numbers: list[int], transform: Callable[[int], int]) -> list[int]:
    return [transform(n) for n in numbers]

print(apply(lambda x: x**2, 5))  # 25
print(apply_many([1, 2, 3, 4, 5], lambda x: x * 3))

## 7. TypeVar — Generic Functions

In [ ]:
from typing import TypeVar

T = TypeVar('T')

def first(items: list[T]) -> T:
    """Return the first element, preserving its type."""
    return items[0]

def last(items: list[T]) -> T:
    return items[-1]

# Type is inferred correctly by mypy
nums = [1, 2, 3]
strs = ['a', 'b', 'c']

num = first(nums)   # T inferred as int
s = first(strs)     # T inferred as str

print(num, type(num))
print(s, type(s))

## 8. Type Aliases

In [ ]:
# Type aliases make complex types readable
from typing import TypeAlias

# Simple aliases
Vector: TypeAlias = list[float]
Matrix: TypeAlias = list[list[float]]
JSON: TypeAlias = dict[str, 'Any']

def dot_product(v1: Vector, v2: Vector) -> float:
    return sum(a * b for a, b in zip(v1, v2))

def transpose(matrix: Matrix) -> Matrix:
    return [list(row) for row in zip(*matrix)]

v1: Vector = [1.0, 2.0, 3.0]
v2: Vector = [4.0, 5.0, 6.0]
print(f'Dot product: {dot_product(v1, v2)}')

m: Matrix = [[1, 2, 3], [4, 5, 6]]
print(f'Transposed: {transpose(m)}')

## 9. Type Hints with Dataclasses

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime

@dataclass
class User:
    name: str
    email: str
    age: int
    is_active: bool = True
    tags: list[str] = field(default_factory=list)
    created_at: datetime = field(default_factory=datetime.now)

u1 = User('Alice', 'alice@example.com', 30)
u2 = User('Bob', 'bob@example.com', 25, tags=['admin', 'dev'])

print(u1)
print(u2)
print(u1.tags)
print(u2.tags)

## Mini-Project: Typed Data Processing Pipeline

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Optional

@dataclass
class Product:
    id: int
    name: str
    price: float
    category: str
    stock: int = 0
    tags: list[str] = field(default_factory=list)

@dataclass
class Cart:
    items: list[tuple[Product, int]] = field(default_factory=list)
    discount: float = 0.0  # 0.0 to 1.0

    def add(self, product: Product, quantity: int = 1) -> None:
        self.items.append((product, quantity))

    def subtotal(self) -> float:
        return sum(p.price * qty for p, qty in self.items)

    def total(self) -> float:
        return self.subtotal() * (1 - self.discount)

    def receipt(self) -> str:
        lines: list[str] = []
        lines.append('=== RECEIPT ===')
        for product, qty in self.items:
            lines.append(f'{product.name:20} x{qty:2}  £{product.price * qty:.2f}')
        lines.append(f'{"Subtotal":25} £{self.subtotal():.2f}')
        if self.discount:
            lines.append(f'{f"Discount ({self.discount:.0%})":25} -£{self.subtotal() * self.discount:.2f}')
        lines.append(f'{"TOTAL":25} £{self.total():.2f}')
        return '\n'.join(lines)


# Create products
laptop = Product(1, 'Laptop Pro', 999.99, 'Electronics', 5)
mouse = Product(2, 'Wireless Mouse', 29.99, 'Electronics', 50)
book = Product(3, 'Python Handbook', 39.99, 'Books', 100)

# Create cart
cart = Cart(discount=0.1)  # 10% discount
cart.add(laptop, 1)
cart.add(mouse, 2)
cart.add(book, 3)

print(cart.receipt())